In [10]:
%pip install murmurhash

Note: you may need to restart the kernel to use updated packages.


In [12]:
import numpy as np
import murmurhash

class FiltroBloom:  
    def __init__(self, n, m, k):  
        self.n = n
        self.m = m
        self.k = k
        self.arrbit = np.zeros(n, dtype=np.bool)
    
    def registra(self, s):
        for i in range(self.k):
            hv = murmurhash.mrmr.hash(s, i) % self.n 
            self.arrbit[hv] = True 

    def verifica(self, s):
        bits = np.zeros(self.k, dtype=np.bool)
        for i in range(self.k):
            hv = murmurhash.mrmr.hash(s,i) % self.n 
            bits[i] = self.arrbit[hv]
        return np.all(bits)

In [13]:
!wget https://gist.githubusercontent.com/demersdesigns/4442cd84c1cc6c5ccda9b19eac1ba52b/raw/cf06109a805b661dd12133f9aa4473435e478569/craft-popular-urls

"wget" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [16]:
import urllib.request

url = 'https://gist.githubusercontent.com/demersdesigns/4442cd84c1cc6c5ccda9b19eac1ba52b/raw/cf06109a805b661dd12133f9aa4473435e478569/craft-popular-urls'

print("Descargando datos directamente a la memoria RAM...")

# 1. Abrimos la URL y leemos los datos directamente sin guardar archivos
with urllib.request.urlopen(url) as respuesta:
    # Leemos los bytes, los decodificamos a texto normal y separamos por saltos de línea
    datos_crudos = respuesta.read().decode('utf-8')
    urls = datos_crudos.split('\n')

# 2. Limpiamos espacios vacíos por si acaso
urls = [u for u in urls if u.strip() != '']

print("¡Extracción completa!")
print(f"Se encontraron {len(urls)} URLs.")
print("\nPrimeras 5 URLs encontradas:")
print(urls[:5])

Descargando datos directamente a la memoria RAM...
¡Extracción completa!
Se encontraron 100 URLs.

Primeras 5 URLs encontradas:
['http://www.youtube.com', 'http://www.facebook.com', 'http://www.baidu.com', 'http://www.yahoo.com', 'http://www.amazon.com']


In [17]:
fb = FiltroBloom(1000, len(urls), 5)

for u in urls:
    fb.registra(u)

In [18]:
print(u'Proporción de bits distintos a 0 = {0}'.format(fb.arrbit.nonzero()[0].size / fb.arrbit.size))

Proporción de bits distintos a 0 = 0.39


In [19]:
print(fb.verifica('http://www.youtube.com'))
print(fb.verifica('http://www.facebook.com'))
print(fb.verifica('http://www.yahoo.com'))
print(fb.verifica('http://www.amazon.com'))
print(fb.verifica('http://www.wikipedia.org'))
print(fb.verifica('http://www.baidu.com'))
print(fb.verifica('http://www.twitter.com'))
print(fb.verifica('http://www.unam.mx'))
print(fb.verifica('http://www.twitter.com/'))
print(fb.verifica('https://www.twitter.com'))
print(fb.verifica('https://www.twitter.com/'))

True
True
True
True
True
True
True
False
False
True
False


# ==========================================
# EJERCICIO: FILTRO DE BLOOM CON MD5 Y EXPLORACIÓN
# ==========================================

In [20]:
import numpy as np
import hashlib

In [21]:
# 1. REESCRITURA DE LA CLASE CON NUEVA FUNCIÓN HASH
class FiltroBloomMD5:  
    def __init__(self, n, m, k):  
        self.n = n
        self.m = m
        self.k = k
        # Nota: Usamos bool nativo de Python para evitar advertencias de NumPy
        self.arrbit = np.zeros(n, dtype=bool)

    # NUEVO MÉTODO: Generador de Hash con MD5
    def _hash_func(self, s, i):
        # Concatenamos la URL con el número de ciclo (ej. "youtube.com_0", "youtube.com_1")
        cadena = f"{s}_{i}".encode('utf-8')
        # Generamos el hash hexadecimal y lo convertimos a un número entero
        hash_hex = hashlib.md5(cadena).hexdigest()
        numero_gigante = int(hash_hex, 16)
        # Lo encerramos en el límite de nuestro arreglo
        return numero_gigante % self.n

    def registra(self, s):
        for i in range(self.k):
            hv = self._hash_func(s, i) 
            self.arrbit[hv] = True 

    def verifica(self, s):
        bits = np.zeros(self.k, dtype=bool)
        for i in range(self.k):
            hv = self._hash_func(s, i) 
            bits[i] = self.arrbit[hv]
        return np.all(bits)

In [22]:
print("--- PRUEBA DE HIPERPARÁMETROS ---")
# Asumimos que la lista 'urls' ya existe en memoria por la celda anterior

# Diferentes escenarios (Tamaño 'n', Hashes 'k')
escenarios = [
    {"n": 200,  "k": 2},   # Poca memoria, pocos hashes
    {"n": 1000, "k": 5},   # Escenario original
    {"n": 1000, "k": 15},  # Muchísimos hashes
    {"n": 5000, "k": 5}    # Muchísima memoria
]

url_trampa = 'https://www.twitter.com' # La URL que causaba el Falso Positivo

for config in escenarios:
    n = config["n"]
    k = config["k"]
    
    # Creamos un filtro nuevo para cada escenario
    fb_prueba = FiltroBloomMD5(n, len(urls), k)
    
    # Llenamos el filtro
    for u in urls:
        fb_prueba.registra(u)
        
    # Calculamos métricas
    proporcion = fb_prueba.arrbit.nonzero()[0].size / fb_prueba.arrbit.size
    falso_positivo = fb_prueba.verifica(url_trampa)
    
    print(f"n={n:<4} | k={k:<2} | Ocupación: {proporcion*100:>5.1f}% | ¿Dio Falso Positivo?: {falso_positivo}")

--- PRUEBA DE HIPERPARÁMETROS ---
n=200  | k=2  | Ocupación:  61.0% | ¿Dio Falso Positivo?: False
n=1000 | k=5  | Ocupación:  39.7% | ¿Dio Falso Positivo?: False
n=1000 | k=15 | Ocupación:  79.2% | ¿Dio Falso Positivo?: False
n=5000 | k=5  | Ocupación:   9.4% | ¿Dio Falso Positivo?: False


### Análisis de Resultados: Modificación del Filtro de Bloom y Hash

**1. Sustitución de la Función Hash (MD5 vs MurmurHash)**
Se reemplazó la librería `murmurhash` por el algoritmo criptográfico **MD5** de la librería nativa `hashlib`. Al concatenar el índice $i$ al string original, logramos simular las $k$ funciones hash necesarias. Un fenómeno destacable de esta ejecución fue la desaparición del Falso Positivo para la URL de prueba. Al cambiar el algoritmo matemático, la "huella digital" cambió por completo, evitando la colisión que sí ocurría con MurmurHash. Esto subraya la naturaleza **probabilística** del filtro: las colisiones dependen enteramente de la distribución matemática subyacente de la función hash elegida.

**2. Impacto de los Hiperparámetros ($n$ y $k$)**
Los resultados del simulador muestran claramente el delicado balance de esta estructura de datos:
* **Saturación por falta de memoria ($n=200, k=2$):** A pesar de usar solo 2 hashes, la memoria es tan pequeña que la ocupación se dispara al **61.0%**. Es ineficiente.
* **El escenario ideal ($n=1000, k=5$):** Mantiene un balance saludable de **39.7%** de ocupación. Suficiente espacio libre para evitar colisiones, pero sin desperdiciar RAM.
* **Saturación por exceso de huellas ($n=1000, k=15$):** Al obligar a cada URL a encender 15 bits, el arreglo se satura a un **79.2%**. Un filtro tan lleno es estadísticamente inútil, ya que casi cualquier consulta futura rebotará contra bits ya encendidos, disparando la tasa de falsos positivos globales.
* **Holgura excesiva ($n=5000, k=5$):** La ocupación cae al **9.4%**. Aunque elimina los falsos positivos casi por completo, se rompe el objetivo principal del Big Data: optimizar recursos. Se está desperdiciando memoria espacial $\mathcal{O}(n)$.

**Conclusión:**
El diseño de un Filtro de Bloom no debe basarse en valores empíricos. Los hiperparámetros $n$ y $k$ deben calcularse mediante fórmulas probabilísticas estrictas, teniendo en cuenta la cantidad de registros esperados ($m$) y la tolerancia al error permitida por la arquitectura del sistema.